# Tuning Veloce della Riduzione Dimensionale (UMAP & t-SNE)

Questo notebook permette di esplorare rapidamente gli iperparametri di UMAP e t-SNE su una matrice di features caricata. 

Vengono fissati i parametri "nested" (`metric`, `regress_out_volume`) e variati gli altri parametri di grid per ottenere curve di metriche (trustworthiness) e griglie di subplot degli embedding risultanti, proprio come nella pipeline di produzione.

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tempfile
from IPython.display import Image, display

# Aggiunge la directory root del progetto al path per importare src
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

# Ripristina il backend interattivo per i plot inline nel notebook
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

from src.utils.artifacts import load_matrix
from src.analysis.reduction import REDUCTION_METHODS, embedding_for_viz
from src.analysis.tuning import run_tuning_sweep, evaluate_umap, evaluate_tsne
from src.analysis.plotting import plot_tuning_curve, plot_embedding_grid_blocks
from src.analysis.params import load_nested_params, load_method_params, load_tuning_grid, load_trustworthiness_n_neighbors

print("Librerie importate con successo e backend matplotlib impostato per visualizzazione inline.")

## 1. Setup dei Percorsi e Caricamento Matrice

Definiamo il percorso della matrice lesionale costruita e carichiamo i dati.

In [ ]:
# Percorso della matrice lesionale di input (modificabile all'occorrenza)
input_matrix_dir = Path("../data/derived/lesion_matrix/21-07_s1.1")

try:
    X, metadata, extra_arrays = load_matrix(input_matrix_dir)
    print(f"Matrice caricata correttamente da: {input_matrix_dir}")
    print(f"Shape di X: {X.shape[0]} soggetti x {X.shape[1]} features")
    print(f"Colonne metadati: {list(metadata.columns)}")
except Exception as e:
    print(f"Errore nel caricamento della matrice: {e}")

## 2. Definizione Configurazione e Parametri (Nested / Grid)

Definiamo la configurazione di tuning. Carichiamo le impostazioni predefinite dal registro del progetto (`config/registry/params_reduction.json`).

In [ ]:
# Carica le configurazioni dal registro
params_file = Path("../config/registry/params_reduction.json")

# Scegli il metodo da testare: 'umap' oppure 'tsne'
reduction_method = "umap"  

base_params, _ = load_method_params(params_file, reduction_method)
tuning_grid = load_tuning_grid(params_file, reduction_method)
nested_params = load_nested_params(params_file, reduction_method, tuning_grid)
trustworthiness_n_neighbors = load_trustworthiness_n_neighbors(params_file, reduction_method)

print(f"Metodo selezionato: {reduction_method}")
print(f"Base params: {base_params}")
print(f"Tuning grid: {tuning_grid}")
print(f"Nested params (quelli strutturali): {nested_params}")
print(f"Vicini per calcolo della trustworthiness: {trustworthiness_n_neighbors}")

## 3. Modifica dei Nested Parameters (Fissati)

Qui fissiamo i parametri nested per questa run di tuning veloce (es. metric = 'jaccard', regress_out_volume = False). Gli altri parametri varieranno secondo la `tuning_grid`.

In [ ]:
# Fissa i parametri nested per il test veloce
fixed_metric = "jaccard"  # "euclidean", "jaccard", "dice"
fixed_regress_volume = False

# Aggiorniamo base_params con i valori fissati
base_params["metric"] = fixed_metric
base_params["regress_out_volume"] = fixed_regress_volume

# Rimuoviamo i parametri nested dalla griglia di sweep in modo che vengano trattati come fissati
sweep_grid = {k: v for k, v in tuning_grid.items() if k not in nested_params}

print(f"Parametri nested fissati per lo sweep:")
print(f" - metric: {fixed_metric}")
print(f" - regress_out_volume: {fixed_regress_volume}")
print(f"Nuova griglia di sweep (free parameters): {sweep_grid}")

## 4. Esecuzione dello Sweep di Tuning

Eseguiamo lo sweep sui parametri rimasti liberi. La funzione `run_tuning_sweep` calcola la trustworthiness per ciascuna combinazione.

In [ ]:
# Eseguiamo lo sweep in memoria
results, embeddings_by_combo = run_tuning_sweep(
    reduction_method, X, base_params, sweep_grid, trustworthiness_n_neighbors
)

# Mostra la tabella dei risultati ordinata per trustworthiness decrescente
results_sorted = results.sort_values(by="trustworthiness", ascending=False)
display(results_sorted)

## 5. Visualizzazione dei Risultati (Curve o Griglie)

A seconda di quanti parametri sono stati variati:
- Se **un solo parametro** è variato (es. solo perplexity per t-SNE), mostriamo la curva di tuning.
- Se **più parametri** sono variati, mostriamo la griglia di subplot degli embedding corrispondenti per ispezionare visivamente lo spazio di embedding.

In [ ]:
free_params = list(sweep_grid.keys())
metric_col = "trustworthiness"

if len(free_params) == 1:
    # 1. Plotta la curva di tuning
    var_name = free_params[0]
    print(f"Plotto la curva di tuning per il parametro variato: {var_name}")
    
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmpfile:
        tmp_path = Path(tmpfile.name)
        
    try:
        title = f"Tuning {reduction_method} ({fixed_metric}, regress_vol={fixed_regress_volume})"
        plot_tuning_curve(results, var_name, metric_col, tmp_path, title)
        display(Image(filename=tmp_path))
    finally:
        if tmp_path.exists():
            tmp_path.unlink()
            
else:
    # 2. Costruisce la griglia degli embedding
    print("Costruisco la griglia degli embedding per i parametri variati...")
    
    # Costruiamo i blocchi per la visualizzazione grid
    keys = list(sweep_grid.keys())
    
    blocks = []
    for varying in free_params:
        held_params = [p for p in free_params if p != varying]
        cells = []
        for value in sweep_grid[varying]:
            combo_values = []
            for key in keys:
                if key == varying:
                    combo_values.append(value)
                else:
                    combo_values.append(base_params[key])
            
            combo = tuple(combo_values)
            if combo not in embeddings_by_combo:
                continue
                
            embedding = embeddings_by_combo[combo]
            reduction_params_for_combo = {**base_params, **dict(zip(keys, combo))}
            
            # Forza la visualizzazione in 2D per il subplot
            viz_emb = embedding_for_viz(
                reduction_method, X, reduction_params_for_combo, embedding, 2
            )
            cells.append((str(value), viz_emb))
        blocks.append((varying, cells))
        
    # Plottiamo la griglia degli embedding
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmpfile:
        tmp_path = Path(tmpfile.name)
        
    try:
        plot_embedding_grid_blocks(
            blocks, 
            tmp_path, 
            f"{reduction_method} dim 1", 
            f"{reduction_method} dim 2", 
            f"Embedding Grid - {reduction_method.upper()} ({fixed_metric})"
        )
        display(Image(filename=tmp_path))
    finally:
        if tmp_path.exists():
            tmp_path.unlink()